# Installing Dependencies


과제를 수행함에 있어 설치해야 하는 의존성 라이브러리 등이 있을 시 아래 코드 셀에 설치 명령을 작성할 것

In [5]:
# 라이브러리 설치 코드 작성
%pip install xgboost lightgbm scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load


과제에서 활용하는 데이터를 Kaggle API 활용 또는 다운로드 후 불러오기를 통해서 다음과 같은 변수에 저장

In [6]:
# 데이터 로드 코드 작성
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# Kaggle API를 사용할 경우 아래 주석을 해제하고 실행하세요
# !kaggle competitions download -c <competition-name>
# !unzip <competition-name>.zip
# ---------------------------------------------------------------

# Train sensor signals
train_bvp   = pd.read_csv('train-bvp.csv')
train_acc   = pd.read_csv('train-acc.csv')
train_hr    = pd.read_csv('train-hr.csv')
train_eda   = pd.read_csv('train-eda.csv')
train_temp  = pd.read_csv('train-temp.csv')
train_ibi   = pd.read_csv('train-ibi.csv')
train_brain = pd.read_csv('train-brain.csv')

# Test sensor signals
test_bvp   = pd.read_csv('test-bvp.csv')
test_acc   = pd.read_csv('test-acc.csv')
test_hr    = pd.read_csv('test-hr.csv')
test_eda   = pd.read_csv('test-eda.csv')
test_temp  = pd.read_csv('test-temp.csv')
test_ibi   = pd.read_csv('test-ibi.csv')
test_brain = pd.read_csv('test-brain.csv')

# Labels
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_LABEL  = pd.read_csv('test-label.csv',  index_col='id')

# Required template variable names
# TRAIN_DATA and TEST_DATA will be the engineered feature matrices
# (built in the next section)

# Fix potential whitespace in test timestamp column
TEST_LABEL['timestamp'] = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float).astype(int)
TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(int)

print("Train label shape :", TRAIN_LABEL.shape)
print("Test  label shape :", TEST_LABEL.shape)
print("\nArousal distribution (train):")
print(TRAIN_LABEL['arousal'].value_counts().sort_index())

Train label shape : (1456, 3)
Test  label shape : (1496, 3)

Arousal distribution (train):
arousal
1     55
2    430
3    554
4    345
5     72
Name: count, dtype: int64


## Step 1: Feature Engineering

In [7]:
# ================================================================
# STEP 1 — Feature Engineering
# 각 레이블 타임스탬프 주변 5초 윈도우에서
# 7개 센서 신호별 통계/주파수 특성을 추출합니다.
# ================================================================

from scipy import stats as sp_stats

WINDOW_MS = 5000   # ±2.5 s window around each label timestamp


def get_window(timestamp, pid, df, window_ms=WINDOW_MS):
    """Return rows of df within [ts - half, ts + half] for the given pid."""
    half = window_ms / 2
    mask = (
        (df['pid'] == pid) &
        (df['timestamp'] >= timestamp - half) &
        (df['timestamp'] <= timestamp + half)
    )
    return df.loc[mask]


# ── BVP ──────────────────────────────────────────────────────────
def bvp_features(vals):
    if len(vals) < 2:
        return {}
    f = {}
    f['bvp_mean']     = np.mean(vals)
    f['bvp_std']      = np.std(vals)
    f['bvp_min']      = np.min(vals)
    f['bvp_max']      = np.max(vals)
    f['bvp_range']    = f['bvp_max'] - f['bvp_min']
    f['bvp_median']   = np.median(vals)
    f['bvp_skew']     = sp_stats.skew(vals)
    f['bvp_kurtosis'] = sp_stats.kurtosis(vals)
    f['bvp_energy']   = np.sum(vals ** 2)
    f['bvp_rms']      = np.sqrt(np.mean(vals ** 2))
    if len(vals) > 4:
        fft = np.abs(np.fft.fft(vals))
        f['bvp_fft_energy']       = float(np.sum(fft ** 2))
        f['bvp_spectral_centroid'] = float(
            np.average(np.arange(len(fft)), weights=fft)
            if fft.sum() > 0 else 0
        )
    return f


# ── ACC ──────────────────────────────────────────────────────────
def acc_features(df):
    if len(df) < 2:
        return {}
    f = {}
    for ax in ['x', 'y', 'z']:
        v = df[ax].values
        f[f'acc_{ax}_mean']  = np.mean(v)
        f[f'acc_{ax}_std']   = np.std(v)
        f[f'acc_{ax}_range'] = np.max(v) - np.min(v)
        f[f'acc_{ax}_rms']   = np.sqrt(np.mean(v ** 2))
    mag = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2).values
    f['acc_magnitude_mean'] = np.mean(mag)
    f['acc_magnitude_std']  = np.std(mag)
    f['acc_magnitude_rms']  = np.sqrt(np.mean(mag ** 2))
    if len(mag) > 2:
        jerk = np.diff(mag)
        f['acc_jerk_mean'] = np.mean(np.abs(jerk))
        f['acc_jerk_std']  = np.std(jerk)
    return f


# ── HR ───────────────────────────────────────────────────────────
def hr_features(vals):
    if len(vals) < 2:
        return {}
    f = {}
    f['hr_mean']   = np.mean(vals)
    f['hr_std']    = np.std(vals)
    f['hr_min']    = np.min(vals)
    f['hr_max']    = np.max(vals)
    f['hr_range']  = f['hr_max'] - f['hr_min']
    f['hr_median'] = np.median(vals)
    if len(vals) > 2:
        d = np.diff(vals)
        f['hr_variability']  = np.std(d)
        f['hr_change_mean']  = np.mean(np.abs(d))
    return f


# ── EDA ──────────────────────────────────────────────────────────
def eda_features(vals):
    if len(vals) < 2:
        return {}
    f = {}
    f['eda_mean']  = np.mean(vals)
    f['eda_std']   = np.std(vals)
    f['eda_min']   = np.min(vals)
    f['eda_max']   = np.max(vals)
    f['eda_range'] = f['eda_max'] - f['eda_min']
    if len(vals) > 2:
        d = np.diff(vals)
        f['eda_phasic_mean'] = np.mean(np.abs(d))
        f['eda_phasic_std']  = np.std(d)
    return f


# ── TEMP ─────────────────────────────────────────────────────────
def temp_features(vals):
    if len(vals) < 2:
        return {}
    return {
        'temp_mean' : np.mean(vals),
        'temp_std'  : np.std(vals),
        'temp_min'  : np.min(vals),
        'temp_max'  : np.max(vals),
        'temp_range': np.max(vals) - np.min(vals),
    }


# ── IBI ──────────────────────────────────────────────────────────
def ibi_features(vals):
    if len(vals) < 2:
        return {}
    f = {}
    f['ibi_mean'] = np.mean(vals)
    f['ibi_std']  = np.std(vals)
    f['ibi_min']  = np.min(vals)
    f['ibi_max']  = np.max(vals)
    f['ibi_cv']   = np.std(vals) / (np.mean(vals) + 1e-6)
    if len(vals) > 2:
        f['ibi_rmssd'] = np.sqrt(np.mean(np.diff(vals) ** 2))
    return f


# ── BRAIN / EEG ──────────────────────────────────────────────────
EEG_BANDS = ['delta', 'lowAlpha', 'highAlpha',
              'lowBeta', 'highBeta', 'lowGamma', 'middleGamma', 'theta']

def brain_features(df):
    if len(df) == 0:
        return {}
    f = {}
    for col in EEG_BANDS:
        log_v = np.log1p(df[col].values)   # log-normalise large raw values
        f[f'brain_{col}_mean']   = np.mean(log_v)
        f[f'brain_{col}_std']    = np.std(log_v)
        f[f'brain_{col}_median'] = np.median(log_v)
    # Neuroscience-motivated band ratios
    d_mu  = np.mean(np.log1p(df['delta'].values))
    th_mu = np.mean(np.log1p(df['theta'].values))
    al_mu = np.mean(np.log1p(((df['lowAlpha'] + df['highAlpha']) / 2).values))
    be_mu = np.mean(np.log1p(((df['lowBeta']  + df['highBeta'])  / 2).values))
    f['brain_delta_theta_ratio'] = d_mu  / (th_mu + 1e-6)
    f['brain_alpha_beta_ratio']  = al_mu / (be_mu + 1e-6)
    return f


# ── Master extraction function ────────────────────────────────────
def extract_all_features(ts, pid,
                          bvp_df, acc_df, hr_df,
                          eda_df, temp_df, ibi_df, brain_df):
    feat = {}
    feat.update(bvp_features  (get_window(ts, pid, bvp_df  )['value'].values))
    feat.update(acc_features  (get_window(ts, pid, acc_df  )))
    feat.update(hr_features   (get_window(ts, pid, hr_df   )['value'].values))
    feat.update(eda_features  (get_window(ts, pid, eda_df  )['value'].values))
    feat.update(temp_features (get_window(ts, pid, temp_df )['value'].values))
    feat.update(ibi_features  (get_window(ts, pid, ibi_df  )['value'].values))
    feat.update(brain_features(get_window(ts, pid, brain_df)))
    return feat


print("Feature extraction functions defined. Running extraction...")

Feature extraction functions defined. Running extraction...


In [8]:
# ── Extract features for TRAIN set ───────────────────────────────
train_records = []
for idx, row in TRAIN_LABEL.reset_index().iterrows():
    feat = extract_all_features(
        row['timestamp'], row['pid'],
        train_bvp, train_acc, train_hr,
        train_eda, train_temp, train_ibi, train_brain
    )
    feat['id'] = row['id']
    train_records.append(feat)

train_feat_df = pd.DataFrame(train_records).set_index('id')
print(f"Train feature matrix: {train_feat_df.shape}")

# ── Extract features for TEST set ────────────────────────────────
test_records = []
for idx, row in TEST_LABEL.reset_index().iterrows():
    feat = extract_all_features(
        row['timestamp'], row['pid'],
        test_bvp, test_acc, test_hr,
        test_eda, test_temp, test_ibi, test_brain
    )
    feat['id'] = row['id']
    test_records.append(feat)

test_feat_df = pd.DataFrame(test_records).set_index('id')
print(f"Test  feature matrix: {test_feat_df.shape}")

Train feature matrix: (1456, 81)
Test  feature matrix: (1496, 81)


## Step 2: Preprocessing

In [9]:
# ================================================================
# STEP 2 — Preprocessing
# ================================================================
from sklearn.preprocessing import StandardScaler

# Align feature columns between train and test
shared_cols = [c for c in train_feat_df.columns if c in test_feat_df.columns]
train_feat_df = train_feat_df[shared_cols]
test_feat_df  = test_feat_df[shared_cols]

# Replace NaN / Inf
train_feat_df = train_feat_df.replace([np.inf, -np.inf], np.nan).fillna(0)
test_feat_df  = test_feat_df.replace([np.inf, -np.inf], np.nan).fillna(0)

# Standardise (fit on train only)
scaler = StandardScaler()
X_train_arr = scaler.fit_transform(train_feat_df.values)
X_test_arr  = scaler.transform(test_feat_df.values)

# Labels: 1-indexed → 0-indexed for XGBoost / LightGBM
y_train = TRAIN_LABEL.loc[train_feat_df.index, 'arousal'].values
y_train_idx = y_train - 1          # 0..4

# ── Class weights (inverse frequency) ────────────────────────────
classes, counts = np.unique(y_train, return_counts=True)
class_weight = {c: len(y_train) / (len(classes) * n)
                for c, n in zip(classes, counts)}
sample_weights = np.array([class_weight[c + 1] for c in y_train_idx])

# ── Required template variables ─────────────────────────────────
TRAIN_DATA = pd.DataFrame(X_train_arr,
                          index=train_feat_df.index,
                          columns=shared_cols)
TEST_DATA  = pd.DataFrame(X_test_arr,
                          index=test_feat_df.index,
                          columns=shared_cols)

print(f"TRAIN_DATA shape : {TRAIN_DATA.shape}")
print(f"TEST_DATA  shape : {TEST_DATA.shape}")
print(f"\nClass weights:")
for c, w in sorted(class_weight.items()):
    print(f"  Level {c}: {w:.4f}")

TRAIN_DATA shape : (1456, 81)
TEST_DATA  shape : (1496, 81)

Class weights:
  Level 1: 5.2945
  Level 2: 0.6772
  Level 3: 0.5256
  Level 4: 0.8441
  Level 5: 4.0444


## Step 3: Model Training (Ensemble — XGBoost + LightGBM)

In [10]:
# ================================================================
# STEP 3 — Model Training
# XGBoost + LightGBM soft-voting ensemble
# Both models receive inverse-frequency sample weights to
# counter the severe class imbalance (Level 1: 3.8%, Level 3: 38%)
# ================================================================
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# ── XGBoost ──────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators      = 300,
    max_depth         = 7,
    learning_rate     = 0.08,
    subsample         = 0.85,
    colsample_bytree  = 0.85,
    gamma             = 0.5,
    min_child_weight  = 1,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'mlogloss',
)
xgb_model.fit(
    TRAIN_DATA.values, y_train_idx,
    sample_weight=sample_weights,
    verbose=False,
)
print("XGBoost training complete.")

# ── LightGBM ─────────────────────────────────────────────────────
lgb_params = dict(
    num_leaves   = 31,
    max_depth    = 7,
    learning_rate= 0.08,
    objective    = 'multiclass',
    num_class    = 5,
    metric       = 'multi_error',
    verbose      = -1,
    random_state = 42,
)
lgb_train_ds = lgb.Dataset(
    TRAIN_DATA.values,
    label=y_train_idx,
    weight=sample_weights,
)
lgb_model = lgb.train(lgb_params, lgb_train_ds, num_boost_round=300)
print("LightGBM training complete.")

XGBoost training complete.
LightGBM training complete.


## Step 4: Cross-Validation

In [11]:
# ================================================================
# STEP 4 — 5-Fold Stratified Cross-Validation
# ================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_xgb, cv_lgb, cv_ens = [], [], []

for fold, (tr_idx, val_idx) in enumerate(
        skf.split(TRAIN_DATA.values, y_train_idx)):

    Xtr, Xval = TRAIN_DATA.values[tr_idx], TRAIN_DATA.values[val_idx]
    ytr, yval = y_train_idx[tr_idx],        y_train_idx[val_idx]
    wtr       = sample_weights[tr_idx]

    # XGBoost fold
    xgb_cv = xgb.XGBClassifier(
        n_estimators=300, max_depth=7, learning_rate=0.08,
        subsample=0.85, colsample_bytree=0.85, gamma=0.5,
        random_state=42, n_jobs=-1, eval_metric='mlogloss'
    )
    xgb_cv.fit(Xtr, ytr, sample_weight=wtr, verbose=False)
    xgb_proba = xgb_cv.predict_proba(Xval)
    cv_xgb.append(accuracy_score(yval, xgb_proba.argmax(axis=1)))

    # LightGBM fold
    lgb_ds_cv = lgb.Dataset(Xtr, label=ytr, weight=wtr)
    lgb_cv    = lgb.train(lgb_params, lgb_ds_cv, num_boost_round=300)
    lgb_proba = lgb_cv.predict(Xval)
    cv_lgb.append(accuracy_score(yval, lgb_proba.argmax(axis=1)))

    # Ensemble fold
    ens_proba = (xgb_proba + lgb_proba) / 2
    cv_ens.append(accuracy_score(yval, ens_proba.argmax(axis=1)))

    print(f"Fold {fold+1}  XGB={cv_xgb[-1]:.4f}  "
          f"LGB={cv_lgb[-1]:.4f}  Ensemble={cv_ens[-1]:.4f}")

print(f"\nMean  XGB={np.mean(cv_xgb):.4f}  "
      f"LGB={np.mean(cv_lgb):.4f}  Ensemble={np.mean(cv_ens):.4f}")
print(f"Std   XGB={np.std(cv_xgb):.4f}  "
      f"LGB={np.std(cv_lgb):.4f}  Ensemble={np.std(cv_ens):.4f}")

Fold 1  XGB=0.6986  LGB=0.7123  Ensemble=0.7158
Fold 2  XGB=0.7251  LGB=0.7182  Ensemble=0.7148
Fold 3  XGB=0.6598  LGB=0.6495  Ensemble=0.6667
Fold 4  XGB=0.7148  LGB=0.7010  Ensemble=0.7113
Fold 5  XGB=0.7457  LGB=0.7457  Ensemble=0.7423

Mean  XGB=0.7088  LGB=0.7054  Ensemble=0.7102
Std   XGB=0.0289  LGB=0.0316  Ensemble=0.0244


## Step 5: Generate Predictions

In [12]:
# ================================================================
# STEP 5 — Final predictions on the test set
# ================================================================

xgb_test_proba = xgb_model.predict_proba(TEST_DATA.values)       # shape (N, 5)
lgb_test_proba = lgb_model.predict(TEST_DATA.values)             # shape (N, 5)

ens_test_proba = (xgb_test_proba + lgb_test_proba) / 2
ens_test_proba /= ens_test_proba.sum(axis=1, keepdims=True)      # normalise

# Convert 0-indexed classes back to 1-indexed arousal levels
final_predictions = ens_test_proba.argmax(axis=1) + 1

print("Prediction distribution:")
unique, cnts = np.unique(final_predictions, return_counts=True)
for u, c in zip(unique, cnts):
    print(f"  Level {u}: {c:4d}  ({100*c/len(final_predictions):.1f}%)")

Prediction distribution:
  Level 1:  119  (8.0%)
  Level 2:  458  (30.6%)
  Level 3:  608  (40.6%)
  Level 4:  238  (15.9%)
  Level 5:   73  (4.9%)


Kaggle 제출물인 **csv 파일**을 생성하는 코드를 아래에 작성할 것

In [13]:
# Kaggle 제출물 생성 코드 작성

YOUR_PREDICTIONS = pd.DataFrame({
    'id'     : TEST_DATA.index,
    'arousal': final_predictions
})

YOUR_PREDICTIONS.to_csv('submission-v1.csv', index=False)

print("submission-v1.csv saved.")
print(YOUR_PREDICTIONS.head(10))

submission-v1.csv saved.
     id  arousal
0  2054        3
1  2055        3
2  2056        3
3  2057        3
4  2058        3
5  2059        4
6  2060        4
7  2061        3
8  2062        4
9  2063        3
